#**Sistema de Gestión de Catálogo - Creaciones Infinitas**
*Este cuaderno automatiza la unificación de listas de proveedores y la preparación de archivos para la carga masiva en Tiendanube.*

##**Estándares de Desarrollo** (Manual de Estilo)
Este proyecto se rige por un Protocolo de Blindaje de Datos para garantizar la estabilidad del negocio:

**Regla 1**: *Persistencia en Drive: No se utilizan rutas locales. Todo archivo se gestiona en la infraestructura de Google Drive bajo la ruta maestra BASE_PATH.*

**Regla 2**: *Interfaz y Auditoría: Cada proceso crítico incluye feedback visual (tqdm) y tablas interactivas (data_table) para validación humana antes de la exportación.*

**Regla 3**: *Integridad y Modularidad: El código se organiza en funciones independientes. Se aplican técnicas de limpieza de strings y normalización de tipos para evitar errores de ejecución en la carga masiva.*

##**Paso 1**: **Configuración y Conexión**
*En esta sección se vincula el entorno con Google Drive y se definen las rutas maestras de trabajo siguiendo el protocolo de persistencia.*

In [4]:
# @title 🛠️ 1. Configuración del Entorno y Estructura de Carpetas
import os
import pandas as pd
import datetime
from google.colab import drive, data_table
from tqdm.auto import tqdm

drive.mount('/content/drive', force_remount=True)
BASE_PATH = '/content/drive/MyDrive/Creaciones_Infinitas'

# Definimos las subcarpetas estratégicas
INPUT_PATH = os.path.join(BASE_PATH, 'materia_prima')
OUTPUT_PATH = os.path.join(BASE_PATH, 'resultados')

# Crear las carpetas si no existen
for carpeta in [INPUT_PATH, OUTPUT_PATH]:
    os.makedirs(carpeta, exist_ok=True)

data_table.enable_dataframe_formatter()

print(f"✅ Entorno listo.")
print(f"📥 Origen (Materia Prima): {INPUT_PATH}")
print(f"📤 Destino (Resultados): {OUTPUT_PATH}")

Mounted at /content/drive
✅ Entorno listo.
📥 Origen (Materia Prima): /content/drive/MyDrive/Creaciones_Infinitas/materia_prima
📤 Destino (Resultados): /content/drive/MyDrive/Creaciones_Infinitas/resultados


##**Paso 2 : Unificación y Normalización**
*Procesamiento de los fragmentos de la LISTA N°77. El sistema consolida los datos, limpia los nombres de las columnas y prepara la base de datos maestra.*

In [7]:
# @title 🔄 2. Unificación y Normalización de Listas
def unificar_y_limpiar_catalogos(ruta_origen, ruta_destino):
    # BUSCAMOS SOLO EN MATERIA PRIMA
    fragmentos = [f for f in os.listdir(ruta_origen) if "LISTA N°77" in f and f.endswith('.csv')]

    if not fragmentos:
        print(f"⚠️ No se encontraron archivos en {ruta_origen}. Mové los 13 CSVs ahí.")
        return None

    print(f"📦 Procesando {len(fragmentos)} archivos de materia prima...")
    listas_temp = [pd.read_csv(os.path.join(ruta_origen, f)) for f in tqdm(fragmentos)]

    df_unificado = pd.concat(listas_temp, ignore_index=True).dropna(how='all')
    df_unificado.columns = df_unificado.columns.str.strip().str.lower()

    # Mapeo para Tiendanube
    df_unificado.rename(columns={'artículo': 'Nombre', 'articulo': 'Nombre'}, inplace=True)

    # Guardamos el Maestro en RESULTADOS
    ruta_maestro = os.path.join(ruta_destino, 'LISTA_77_MAESTRA_LIMPIA.csv')
    df_unificado.to_csv(ruta_maestro, index=False)

    print(f"✅ Lista maestra generada con {len(df_unificado)} registros en la carpeta 'resultados'.")
    return df_unificado

df_maestro = unificar_y_limpiar_catalogos(INPUT_PATH, OUTPUT_PATH)

📦 Procesando 13 archivos de materia prima...


  0%|          | 0/13 [00:00<?, ?it/s]

✅ Lista maestra generada con 904 registros en la carpeta 'resultados'.


##**📦 Paso 3: Generación de Archivo de Carga**
*Creación del archivo CSV final optimizado para Tiendanube. Incluye un sistema de versionado por fecha para mantener un historial de actualizaciones.*

In [8]:
# @title 🚀 3. Exportación para Tiendanube (con Historial)
def exportar_tiendanube(df, ruta_destino):
    fecha_hoy = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
    nombre_archivo = f'SUBIDA_TIENDANUBE_{fecha_hoy}.csv'

    ruta_historial = os.path.join(ruta_destino, nombre_archivo)
    ruta_fija = os.path.join(ruta_destino, 'SUBIDA_TIENDANUBE_FINAL.csv')

    df.to_csv(ruta_historial, index=False)
    df.to_csv(ruta_fija, index=False)

    print(f"🎉 Archivos generados con éxito en: {ruta_destino}")
    print(f"📄 Historial: {nombre_archivo}")

if 'df_maestro' in locals() and df_maestro is not None:
    exportar_tiendanube(df_maestro, OUTPUT_PATH)

🎉 Archivos generados con éxito en: /content/drive/MyDrive/Creaciones_Infinitas/resultados
📄 Historial: SUBIDA_TIENDANUBE_2026-04-18_17-42.csv


*"Espacio para futuras integraciones: Cálculo de márgenes, control de stock por SKU, o integración con la API de Tiendanube."*